In [2]:
import pandas as pd
import numpy as np

In [5]:
data=pd.read_csv('grocery_data_final_structured.csv')
data.head()

,received_date,lpo,in_season,product,product_id,category,sub_category,shelf_life_days,maximum_days_on_sale,unit_of_measurement,...,cost_price,waste_cost,spoilage_risk_score,heatwave_flag,storm_flag,shelf_life_reduction_pct,adjusted_days_to_expiry,weather_adjusted_spoilage_risk,weather_adjusted_waste_cost,ipo
0,2024-09-17,2022-12-01,0,Green Tea Bags,1740500|P,Beverages,Tea,540,180,unit,...,78.000144,25.519455,Low,0,0,0.0,540,0.001848,25.519455,232
1,2024-04-03,2022-12-01,0,Oatmeal,1324805|P,Pantry,Breakfast Foods,365,90,lb,...,29.569268,30.377171,Low,0,0,0.0,365,0.002732,30.377171,419
2,2024-06-28,2022-12-05,0,Mango,1756181|P,Fresh Foods,Fruits,7,3,unit,...,91.119522,72485.579503,Low,0,0,0.0,7,0.125000,72485.579503,6438
3,2024-01-27,2022-12-09,0,Onion,1184964|P,Fresh Foods,Vegetables,30,10,lb,...,45.406940,3119.896183,Low,0,0,0.0,30,0.032258,3119.896183,2219
4,2024-02-24,2022-12-05,0,Brown Rice,1737567|P,Pantry,Grains & Rice,180,60,lb,...,11.177385,2.593647,Low,0,0,0.0,180,0.005525,2.593647,81


In [ ]:
# added synthetic weather data

In [11]:
np.random.seed(42)

# Generate realistic weather values
df["temperature"] = np.random.normal(loc=30, scale=5, size=len(df))  # avg 30°C
df["humidity"] = np.random.normal(loc=65, scale=10, size=len(df))    # avg 65%
df["rainfall_mm"] = np.random.exponential(scale=2, size=len(df))

# Heatwave flag
df["heatwave_flag"] = np.where(df["temperature"] > 35, 1, 0)

# Weather risk score (scaled 0-1)
df["weather_risk_score"] = (
    (df["temperature"] / 50) +
    (df["humidity"] / 100)
) / 2

In [ ]:
# added consumer behaviour layer

In [12]:
# Generate synthetic consumer segments
segments = ["Budget", "Premium", "Health-Conscious"]
df["customer_segment"] = np.random.choice(segments, size=len(df))

# Promotion response score
df["discount_response_score"] = np.random.uniform(0.1, 1.0, size=len(df))

# Organic preference flag
df["organic_preference_flag"] = np.random.choice([0,1], size=len(df), p=[0.7,0.3])

# Ready-to-eat preference
df["ready_to_eat_preference"] = np.random.choice([0,1], size=len(df), p=[0.6,0.4])

In [17]:
df.columns = df.columns.str.strip()          # remove spaces
df.columns = df.columns.str.lower()          # make lowercase
df.columns = df.columns.str.replace(" ", "_")  # replace spaces with _

In [20]:
df.rename(columns={"quantity": "units_sold"}, inplace=True)


In [14]:
# created freshness features

In [22]:
# Convert date columns (if not already)
df["expiry_date"] = pd.to_datetime(df["expiry_date"])
df["date"] = pd.to_datetime(df["date"])

# Days to expiry
df["days_to_expiry"] = (df["expiry_date"] - df["date"]).dt.days

# Spoilage risk score
df["spoilage_risk_score"] = (
    (1 / (df["days_to_expiry"] + 1)) * 0.5 +
    df["weather_risk_score"] * 0.5
)

# Waste risk flag
df["waste_risk_flag"] = np.where(
    (df["stock_quantity"] > 50) & (df["days_to_expiry"] < 3),
    1, 0
)

In [15]:
# create demand features

In [23]:
# Sort properly
df = df.sort_values(["store_id", "product_id", "date"])

# Rolling demand (7-day avg)
df["sales_velocity"] = (
    df.groupby(["store_id", "product_id"])["units_sold"]
    .transform(lambda x: x.rolling(7, min_periods=1).mean())
)

# Stockout risk
df["stockout_risk_score"] = df["sales_velocity"] / (df["stock_quantity"] + 1)

KeyError: 'store_id'

In [16]:
# AI decision recommendation column

In [ ]:
def recommend_action(row):
    if row["waste_risk_flag"] == 1:
        return "Apply 20% discount"
    elif row["stockout_risk_score"] > 0.8:
        return "Increase replenishment"
    elif row["heatwave_flag"] == 1 and row["category"] == "Leafy Greens":
        return "Reduce inventory by 15%"
    else:
        return "No action needed"

df["recommended_action"] = df.apply(recommend_action, axis=1)

In [6]:
df=data.copy()

In [7]:
df.shape

(100192, 43)

In [ ]:
# There are 100192 rows and 43 columns

In [8]:
df.describe

<bound method NDFrame.describe of        received_date         lpo  in_season         product product_id  \
0         2024-09-17  2022-12-01          0  Green Tea Bags  1740500|P   
1         2024-04-03  2022-12-01          0         Oatmeal  1324805|P   
2         2024-06-28  2022-12-05          0           Mango  1756181|P   
3         2024-01-27  2022-12-09          0           Onion  1184964|P   
4         2024-02-24  2022-12-05          0      Brown Rice  1737567|P   
...              ...         ...        ...             ...        ...   
100187    2024-04-26  2025-09-08          0    Basmati Rice  1800769|P   
100188    2024-01-09  2025-09-16          1    Strawberries  1486934|P   
100189    2024-06-23  2025-09-17          1          Tomato  1449441|P   
100190    2024-05-18  2025-09-13          0         Ketchup  1083255|P   
100191    2024-07-10  2025-09-18          1           Lemon  1247564|P   

           category     sub_category  shelf_life_days  maximum_days_on_sale  

### Analytical objectives (Business questions)

In [9]:
df.columns

Index(['received_date', 'lpo', 'in_season', 'product', 'product_id',
       'category', 'sub_category', 'shelf_life_days', 'maximum_days_on_sale',
       'unit_of_measurement', 'supplier_rating', 'supplier', 'supplier_id',
       'distance_km', 'moq', 'storage_recommendation',
       'temperature_classification', 'precipitation_classification',
       'wind_classification', 'weather_severity', 'day_classification',
       'is_holiday', 'is_weekend', 'sales_demand', 'sales_volume', 'lead_time',
       'min_stock', 'max_stock', 'stock_quantity', 'date', 'expiry_date',
       'days_to_expiry', 'spoilage_risk', 'cost_price', 'waste_cost',
       'spoilage_risk_score', 'heatwave_flag', 'storm_flag',
       'shelf_life_reduction_pct', 'adjusted_days_to_expiry',
       'weather_adjusted_spoilage_risk', 'weather_adjusted_waste_cost', 'ipo'],
      dtype='object')

In [10]:
df.dtypes

received_date                      object
lpo                                object
in_season                           int64
product                            object
product_id                         object
category                           object
sub_category                       object
shelf_life_days                     int64
maximum_days_on_sale                int64
unit_of_measurement                object
supplier_rating                     int64
supplier                           object
supplier_id                        object
distance_km                         int64
moq                                 int64
storage_recommendation             object
temperature_classification         object
precipitation_classification       object
wind_classification                object
weather_severity                   object
day_classification                 object
is_holiday                          int64
is_weekend                          int64
sales_demand                      